# Financial News Sentiment Analyser

## A tool that reads real financial news headlines and automatically classifies them as positive, negative or neutral for a given stock.

### AI application used by hedge funds and investment banks. It's called NLP-based sentiment analysis

## What to learn and use:

✅ Natural Language Processing (NLP)
✅ Hugging Face Transformers
✅ AI API integration in Python
✅ Sentiment-driven investment signals
✅ Real financial AI application

## Why sentiment analysis of news headlines would be valuable to a portfolio manager?

**Predictive Power:** It helps in forecasting market movements by interpreting the emotional tone of news articles, which can indicate potential stock price increases or declines.

**Risk Management:** It enables early identification of potential risks by monitoring news feeds for unanticipated events or negative sentiment, allowing portfolio managers to adjust their strategies proactively.

**Advanced Trading Strategies:** It integrates sentiment data into trading algorithms, enhancing decision-making processes and potentially leading to higher returns.

**Real-Time Analysis:** Advances in natural language processing and machine learning make real-time sentiment analysis a reality, allowing for immediate reactions to market news.

These insights can help portfolio managers make informed decisions, navigate market fluctuations and optimize their investment strategies effectively.

## Installing the AI Libraries

This installs:

- **Transformers** - Hugging Face's library. Home of 100,000+ pre-trained AI models
- **Torch** - PyTorch, the engine that runs the models
- **Newspaper3k** - scrapes real news articles from the web
- **Sentencepiece** - tokenizer used by many language models

In [ ]:
!pip install transformers torch newspaper3k sentencepiece

## Loading a Pre-Trained Financial AI Model

Core concept: Instead of training an AI model from scratch - which would take weeks and massive computing power - I use a model someone already trained on millions of financial texts.

In [ ]:
from transformers import pipeline

# Load FinBERT - an AI model trained specifically on financial text
# It understands finance language better than general models
print("Loading FinBERT model...")

sentiment_pipeline = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert"
)

print("Model loaded successfully.")
print("This model was trained on financial news, earnings calls and analyst reports.")

## Test It On Real Headlines

In [ ]:
# Real financial headlines - mix of positive, negative, neutral
headlines = [
    "Apple reports record quarterly earnings, beating analyst expectations",
    "Tesla stock crashes after CEO sells $4 billion in shares",
    "Federal Reserve holds interest rates steady amid inflation concerns",
    "Google announces $70 billion share buyback program",
    "Silicon Valley Bank collapses in largest bank failure since 2008",
    "Microsoft Azure cloud revenue grows 28% year over year",
    "Amazon faces antitrust investigation by European regulators",
    "JPMorgan upgrades Apple to overweight with $220 price target",
    "Oil prices surge 8% following OPEC production cut announcement",
    "Emerging market currencies slide as dollar strengthens"
]

# Run each headline through the AI model
results = sentiment_pipeline(headlines)

# Display results
import pandas as pd

sentiment_df = pd.DataFrame({
    'Headline': headlines,
    'Sentiment': [r['label'] for r in results],
    'Confidence': [round(r['score'], 4) for r in results]
})

print(sentiment_df.to_string(index=False))

## Visualising the Sentiment Distribution

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Colour map
color_map = {'positive': '#4CAF82', 'negative': '#E05C5C', 'neutral': '#C9A84C'}
colors = [color_map[s.lower()] for s in sentiment_df['Sentiment']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0A0A0B')

# - LEFT: Confidence per headline
ax1 = axes[0]
bars = ax1.barh(
    range(len(headlines)),
    sentiment_df['Confidence'],
    color=colors, alpha=0.85
)
ax1.set_yticks(range(len(headlines)))
ax1.set_yticklabels(
    [h[:45] + '...' if len(h) > 45 else h for h in headlines],
    fontsize=8, color='#8888AA'
)
ax1.set_xlabel('Model Confidence', color='#8888AA')
ax1.set_title('FinBERT Sentiment Analysis - Financial Headlines',
              color='white', fontsize=11)
ax1.set_facecolor('#111114')
ax1.tick_params(colors='#8888AA')
ax1.set_xlim(0, 1)

# Legend
patches = [
    mpatches.Patch(color='#4CAF82', label='Positive'),
    mpatches.Patch(color='#E05C5C', label='Negative'),
    mpatches.Patch(color='#C9A84C', label='Neutral')
]
ax1.legend(handles=patches, facecolor='#111114',
           labelcolor='white', fontsize=9)

# - RIGHT: Sentiment distribution pie
ax2 = axes[1]
counts = sentiment_df['Sentiment'].str.lower().value_counts()
pie_colors = [color_map.get(s, '#888888') for s in counts.index]
ax2.pie(
    counts.values,
    labels=counts.index.str.capitalize(),
    autopct='%1.0f%%',
    colors=pie_colors,
    startangle=140,
    textprops={'color': 'white', 'fontsize': 11}
)
ax2.set_title('Sentiment Distribution', color='white', fontsize=11)
ax2.set_facecolor('#0A0A0B')

plt.tight_layout()
plt.show()

## Connecting Sentiment to the Portfolio

Making it financially useful, by scoring each stock based on news sentiment.

In [ ]:
# Assign headlines to stocks (simulating a real news feed)
stock_headlines = {
    'AAPL': [
        "Apple reports record quarterly earnings, beating analyst expectations",
        "JPMorgan upgrades Apple to overweight with $220 price target",
    ],
    'TSLA': [
        "Tesla stock crashes after CEO sells $4 billion in shares",
    ],
    'GOOGL': [
        "Google announces $70 billion share buyback program",
        "Amazon faces antitrust investigation by European regulators",
    ],
    'JPM': [
        "Federal Reserve holds interest rates steady amid inflation concerns",
        "JPMorgan upgrades Apple to overweight with $220 price target",
    ],
    'MSFT': [
        "Microsoft Azure cloud revenue grows 28% year over year",
    ]
}

# Score each stock
stock_scores = {}

for ticker, ticker_headlines in stock_headlines.items():
    ticker_results = sentiment_pipeline(ticker_headlines)

    scores = []
    for r in ticker_results:
        if r['label'].lower() == 'positive':
            scores.append(r['score'])
        elif r['label'].lower() == 'negative':
            scores.append(-r['score'])
        else:
            scores.append(0)

    stock_scores[ticker] = {
        'Avg_Sentiment_Score': round(sum(scores) / len(scores), 4),
        'Headlines_Analysed': len(ticker_headlines),
        'Signal': 'BUY' if sum(scores)/len(scores) > 0.2
                  else 'SELL' if sum(scores)/len(scores) < -0.2
                  else 'HOLD'
    }

scores_df = pd.DataFrame(stock_scores).T
print("\n=== SENTIMENT-BASED TRADING SIGNALS ===")
print(scores_df)

## Sentiment Dashboard

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0A0A0B')

scores = scores_df['Avg_Sentiment_Score'].astype(float)
bar_colors = ['#4CAF82' if s > 0.2 else '#E05C5C' if s < -0.2
              else '#C9A84C' for s in scores]

bars = ax.bar(scores_df.index, scores, color=bar_colors, alpha=0.85)

# Signal labels on bars
for bar, (ticker, row) in zip(bars, scores_df.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.01 if bar.get_height() >= 0 else bar.get_height() - 0.04,
        row['Signal'],
        ha='center', color='white',
        fontsize=10, fontweight='bold'
    )

ax.axhline(y=0.2,  color='#4CAF82', linestyle='--',
           alpha=0.5, label='Buy threshold')
ax.axhline(y=-0.2, color='#E05C5C', linestyle='--',
           alpha=0.5, label='Sell threshold')
ax.axhline(y=0,    color='white', linestyle='-', alpha=0.2)

ax.set_facecolor('#111114')
ax.set_title('AI Sentiment Signals by Stock',
             color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Stock', color='#8888AA')
ax.set_ylabel('Sentiment Score', color='#8888AA')
ax.tick_params(colors='#8888AA')
ax.legend(facecolor='#111114', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.show()

## Production-Grade Sentiment Analysis

Building a more honest version that shows confidence intervals and flags low-sample warnings.

In [ ]:
def analyse_stock_sentiment(ticker, headlines, min_headlines=5):
    """
    Production-grade sentiment analyser with confidence warnings
    """
    if len(headlines) == 0:
        return None

    results = sentiment_pipeline(headlines)

    scores = []
    breakdown = {'positive': 0, 'negative': 0, 'neutral': 0}

    for r in results:
        label = r['label'].lower()
        confidence = r['score']
        breakdown[label] += 1

        if label == 'positive':
            scores.append(confidence)
        elif label == 'negative':
            scores.append(-confidence)
        else:
            scores.append(0)

    avg_score = sum(scores) / len(scores)
    confidence_level = "HIGH" if len(headlines) >= min_headlines else \
                       "MEDIUM" if len(headlines) >= 3 else "LOW ⚠️"

    signal = 'BUY' if avg_score > 0.2 else \
             'SELL' if avg_score < -0.2 else 'HOLD'

    # Contrarian flag — strong negative sentiment can mean buy opportunity
    contrarian_flag = "⚡ CONTRARIAN OPPORTUNITY" \
                      if avg_score < -0.5 else ""

    return {
        'Ticker': ticker,
        'Signal': signal,
        'Avg_Score': round(avg_score, 4),
        'Headlines_Analysed': len(headlines),
        'Data_Confidence': confidence_level,
        'Positive_Headlines': breakdown['positive'],
        'Negative_Headlines': breakdown['negative'],
        'Neutral_Headlines': breakdown['neutral'],
        'Contrarian_Flag': contrarian_flag
    }

# Test with expanded headlines
expanded_headlines = {
    'AAPL': [
        "Apple reports record quarterly earnings beating expectations",
        "iPhone 16 demand surges in Asian markets",
        "Apple services revenue hits all time high",
        "Warren Buffett increases Berkshire stake in Apple",
        "Apple announces $110 billion share buyback",
    ],
    'TSLA': [
        "Tesla stock crashes after CEO sells $4 billion in shares",
        "Tesla misses delivery targets for third consecutive quarter",
        "Tesla faces increasing competition from Chinese EV makers",
    ],
    'GOOGL': [
        "Google announces $70 billion share buyback program",
        "Alphabet AI division reports breakthrough in quantum computing",
    ],
    'JPM': [
        "JPMorgan posts record profit driven by investment banking fees",
        "Federal Reserve holds rates steady benefiting bank margins",
        "JPMorgan expands African operations with Kenya headquarters",
    ],
    'MSFT': [
        "Microsoft Azure cloud revenue grows 28% year over year",
        "Microsoft Copilot AI integration drives enterprise adoption",
        "Microsoft acquires AI startup for $1.5 billion",
        "Microsoft named most valuable company globally",
    ]
}

# Run enhanced analysis
enhanced_results = []
for ticker, headlines in expanded_headlines.items():
    result = analyse_stock_sentiment(ticker, headlines)
    if result:
        enhanced_results.append(result)

enhanced_df = pd.DataFrame(enhanced_results)
print("=== ENHANCED SENTIMENT ANALYSIS WITH CONFIDENCE RATINGS ===\n")
print(enhanced_df[[
    'Ticker', 'Signal', 'Avg_Score',
    'Headlines_Analysed', 'Data_Confidence', 'Contrarian_Flag'
]].to_string(index=False))

## Combining Sentiment with Quantitative Data

This is where it gets powerful: combine the AI signal with quantitative analysis (Sharpe Ratio).

In [ ]:
# Portfolio data with Sharpe Ratio
portfolio_data = pd.DataFrame({
    'Ticker':        ['AAPL', 'MSFT', 'GOOGL', 'TSLA', 'JPM'],
    'Sharpe_Ratio':  [1.93,   2.10,   1.90,    0.82,   1.12],
    'Volatility':    [0.224,  0.246,  0.281,   0.584,  0.198],
    'Current_Weight':[0.20,   0.20,   0.15,    0.05,   0.20]
})

# Merge with sentiment scores
sentiment_scores = enhanced_df[['Ticker', 'Avg_Score', 'Signal', 'Data_Confidence']]
combined = portfolio_data.merge(sentiment_scores, on='Ticker')

# Combined score — Sharpe quality + AI sentiment signal
combined['Combined_Score'] = (
    combined['Sharpe_Ratio'] * 0.6 +   # 60% weight to fundamentals
    combined['Avg_Score'] * 2 * 0.4     # 40% weight to sentiment
).round(4)

# Suggested action
def suggest_action(row):
    if row['Signal'] == 'BUY' and row['Sharpe_Ratio'] > 1.5:
        return '🟢 INCREASE WEIGHT'
    elif row['Signal'] == 'SELL' and row['Sharpe_Ratio'] < 1.0:
        return '🔴 REDUCE WEIGHT'
    elif row['Signal'] == 'SELL' and row['Sharpe_Ratio'] > 1.5:
        return '🟡 HOLD — strong fundamentals override sentiment'
    else:
        return '⚪ MAINTAIN'

combined['Suggested_Action'] = combined.apply(suggest_action, axis=1)

print("\n=== QUANT + AI COMBINED DECISION FRAMEWORK ===\n")
print(combined[[
    'Ticker', 'Sharpe_Ratio', 'Signal',
    'Combined_Score', 'Suggested_Action'
]].to_string(index=False))

## Final Visualization: Quant + AI Framework

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('#0A0A0B')

# Color by signal
colors = {'BUY': '#4CAF82', 'SELL': '#E05C5C', 'HOLD': '#C9A84C'}
scatter_colors = [colors[signal] for signal in combined['Signal']]

scatter = ax.scatter(
    combined['Avg_Score'],
    combined['Sharpe_Ratio'],
    s=combined['Current_Weight'] * 3000,
    alpha=0.7,
    c=scatter_colors,
    edgecolors='white',
    linewidth=2
)

# Annotations
for idx, row in combined.iterrows():
    ax.annotate(
        row['Ticker'],
        (row['Avg_Score'], row['Sharpe_Ratio']),
        fontsize=11, fontweight='bold', color='white',
        ha='center', va='center'
    )

# Threshold lines
ax.axvline(x=0.2, color='#4CAF82', linestyle='--', alpha=0.4, linewidth=2)
ax.axvline(x=-0.2, color='#E05C5C', linestyle='--', alpha=0.4, linewidth=2)
ax.axhline(y=1.5, color='#8888AA', linestyle='--', alpha=0.3, linewidth=1.5, label='Good Sharpe (1.5)')

ax.set_facecolor('#111114')
ax.set_xlabel('Sentiment Score (Negative ← → Positive)', color='white', fontsize=12)
ax.set_ylabel('Sharpe Ratio (Risk-Adjusted Return)', color='white', fontsize=12)
ax.set_title('Quant + AI Framework: Sharpe Ratio vs Sentiment Score', 
             color='white', fontsize=14, fontweight='bold', pad=20)
ax.tick_params(colors='#8888AA')
ax.grid(alpha=0.1, color='white')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4CAF82', label='BUY Signal'),
    Patch(facecolor='#E05C5C', label='SELL Signal'),
    Patch(facecolor='#C9A84C', label='HOLD Signal')
]
ax.legend(handles=legend_elements, facecolor='#111114', labelcolor='white', fontsize=11)

plt.tight_layout()
plt.show()

## Summary: What We Built - AI ENGINEERING

✅ **Pre-trained model deployment**
- Loaded FinBERT via Hugging Face
- 110M parameter transformer model

✅ **NLP pipeline**
- Text classification on financial data
- Confidence scoring per prediction

✅ **Production-grade design**
- Sample size warnings
- Contrarian signal detection
- Data confidence ratings

✅ **Quant + AI integration**
- Merged sentiment with Sharpe Ratio
- Weighted decision framework
- Actionable portfolio signals

### Key Insights:

**On Tesla headline:** The model correctly classifies negative news as negative. But a smart investor might see negative sentiment as a buying opportunity (contrarian investing).

**On sample size:** Using only 1-2 headlines per stock creates sample bias. A production system analyzes hundreds of headlines daily.

**On ideal position:** High Sharpe + Positive sentiment = ideal position. High risk-adjusted returns with positive news flow.

**vs Human analyst:** This system processes hundreds of headlines in seconds with zero emotional bias, versus a human reading maybe 20 headlines and being influenced by mood and recency bias.